In [1]:
import sys
from pathlib import Path

# 1. Определяем корень проекта
# (подбираем количество .parent, чтобы попасть в max_projects)
project_root = Path.cwd().parent

# 2. Добавляем корень в пути поиска модулей
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# 3. Проверяем, что путь добавлен
print(f"✅ Project root: {project_root}")
print(f"✅ Sys path: {sys.path[:3]}...")

✅ Project root: c:\Users\123\Desktop\start_vector
✅ Sys path: ['c:\\Users\\123\\Desktop\\start_vector', 'C:\\Python314\\python314.zip', 'C:\\Python314\\DLLs']...


In [2]:
import logging
import pandas as pd
import aiohttp
import asyncio
from datetime import datetime, timedelta
import requests
import json
import time

from src_oop.core.scraper import HTTPClient
from src_oop.core.utils_general import load_api_tokens, load_sima_land_tokens

In [4]:
class SimaLandClient:
    def __init__(self, token=None):
        if token:
            self.token = token
        else:
            self.token = load_sima_land_tokens()['ВЕКТОР']
        self.headers = {
                "x-api-key": self.token
            }

    def _download_all_categories_to_json(self):
        url = "https://www.sima-land.ru/api/v3/category/"
        headers = {"x-api-key": self.token}
        
        all_categories = []
        page = 1
        per_page = 50 # Оптимально для API
        
        print("🚀 Начинаю полную выгрузку категорий. Это может занять пару минут...")
        
        while True:
            params = {
                "page": page,
                "per-page": per_page,
                "is_active": 1 # Берем только те, где есть товары
            }
            
            try:
                response = requests.get(url, headers=headers, params=params)
                response.raise_for_status()
                data = response.json()
                
                items = data.get('items', [])
                if not items:
                    break
                
                # Сохраняем только самое важное, чтобы файл не весил 100Мб
                for item in items:
                    all_categories.append({
                        "id": item['id'],
                        "name": item['name'],
                        "level": item['level'],
                        "path": item['path'] # Поможет понять вложенность
                    })
                
                print(f"✅ Загружено страниц: {page}", end="\r")
                
                # Проверка на последнюю страницу
                if page >= data.get('_meta', {}).get('pageCount', 0):
                    break
                    
                page += 1
                time.sleep(0.05) # Минимальная задержка
                
            except Exception as e:
                print(f"\n❌ Ошибка на странице {page}: {e}")
                break

        # Сохраняем в файл
        with open("sima_categories.json", "w", encoding="utf-8") as f:
            json.dump(all_categories, f, ensure_ascii=False, indent=4)
        
        print(f"\n\nГотово! Всего собрано категорий: {len(all_categories)}")
        print("Данные сохранены в файл: sima_categories.json")

    def get_sim_land_items(self, category_id):
        url = "https://www.sima-land.ru/api/v3/item/"
        all_items = []
        last_id = 0  
        
        params = {
            "category_id": category_id,
            "has_balance": 1,
            "is_disabled": 0,
            "is_deleted": 0,
            "has_photo": 1,
            "per-page": 50,
            "sort": "id"  
        }

        print(f"🚀 Начинаю выгрузку товаров для категории {category_id}...")

        while True:
            # Используем альтернативную пагинацию
            params["id-greater-than"] = last_id
            
            try:
                response = requests.get(url, headers=self.headers, params=params)
                response.raise_for_status()
                data = response.json()
                
                items = data.get('items', [])
                if not items:
                    break     

                for item in items:
                    clean_item = {
                        "sid": item.get('sid'),
                        "name": item.get('name'),
                        "balance": item.get('balance', 0),
                        "price": item.get('price', 0),
                        "price_max": item.get('price_max', 0),
                        "boxtype_id": item.get('boxtype_id'),
                        "box_depth": item.get('box_depth', 0),
                        "box_width": item.get('box_width', 0),
                        "box_height": item.get('box_height', 0),
                        "width": item.get('width', 0),
                        "height": item.get('height', 0),    
                        "per_package": item.get('per_package', 0),
                        "photo_url": item.get('photoUrl', ''),
                    }
                    all_items.append(clean_item)
                
                # Обновляем last_id значением ID последнего товара в текущей пачке
                last_id = items[-1]['id']

                print(f"✅ Загружено товаров: {len(all_items)} (последний ID: {last_id})", end="\r")
                
                # Если пришло меньше, чем мы просили (per-page), значит товары кончились
                if len(items) < params["per-page"]:
                    break
                    
                time.sleep(0.1)
                
            except Exception as e:
                print(f"\n❌ Ошибка: {e}")
                break

        print(f"\nВыгрузка завершена. Итого: {len(all_items)} шт.")
        return all_items

In [ ]:
{"wilds": [{"wild": "wild1105", "count": 14}, {"wild": "wild1398", "count": 76}, {"wild": "wild1542", "count": 48}, {"wild": "wild1569", "count": 28}, {"wild": "wild1592", "count": 9}, {"wild": "wild1603", "count": 26}, {"wild": "wild1604", "count": 31}, {"wild": "wild1973", "count": 24}, {"wild": "wild128", "count": 58}, {"wild": "wild150", "count": 62}, {"wild": "wild151", "count": 65}, {"wild": "wild1556", "count": 26}, {"wild": "wild1594", "count": 2}, {"wild": "wild1607", "count": 9}, {"wild": "wild161", "count": 225}, {"wild": "wild1978", "count": 126}, {"wild": "wild300", "count": 1}, {"wild": "wild312", "count": 29}, {"wild": "wild697", "count": 4}, {"wild": "wild1635", "count": 2}, {"wild": "wild1752", "count": 5}, {"wild": "wild1787", "count": 9}, {"wild": "wild457", "count": 9}, {"wild": "wild1549", "count": 1}, {"wild": "wild160", "count": 6}, {"wild": "wild935", "count": 4}, {"wild": "wild1639", "count": 5}, {"wild": "wild1567", "count": 42}, {"wild": "wild1591", "count": 2}, {"wild": "wild172", "count": 43}, {"wild": "wild1971", "count": 16}, {"wild": "wild1704", "count": 2}, {"wild": "wild252", "count": 4}, {"wild": "wild1106", "count": 8}, {"wild": "wild1675", "count": 2}, {"wild": "wild351", "count": 2}, {"wild": "wild654", "count": 5}, {"wild": "wild1630", "count": 1}, {"wild": "wild359", "count": 1}, {"wild": "wild1625", "count": 25}], "supply_ids": [{"account": "СТАРТ4040", "order_ids": [5003460068, 5004725078, 5004160743, 5004435470, 5004539066, 5004636010, 5005721544, 5005755297, 5005796774, 5004009401, 5003915061, 5004213132, 5004497043, 5004523054, 5004739683, 5005368531, 5003912286, 5004613011, 5005339608, 5004498001, 5005449514, 5005521539], "supply_id": "WB-GI-235725651"}, {"account": "Пилосян", "order_ids": [5003537569, 5003869570, 5004225867, 5004025816, 5004582214, 5004674469, 5004854384, 5005004615, 5005092856, 5005226587, 5005245557, 5005245559, 5005245558, 5005324337, 5005480444, 5005852243, 5004138918, 5003857716, 5004129651, 5004516165, 5004565281, 5004838617, 5005526014, 5003995536, 5004519170, 5005033903, 5005308997, 5005349395, 5005559393, 5005627480, 5005643643, 5005652120, 5005660626, 5003892627, 5003921760, 5004089050, 5004090095, 5004111387, 5004118093, 5004143803, 5004304064, 5004346220, 5004374375, 5004447746, 5004488586, 5004531432, 5004534332, 5004547690, 5004556655, 5004607815, 5004621782, 5004708107, 5004708109, 5004708105, 5004708108, 5004708110, 5004708114, 5004708115, 5004708112, 5004708111, 5004708113, 5004784374, 5004785668, 5004797010, 5004874811, 5005065227, 5005178577, 5005206967, 5005236941, 5005247616, 5005334823, 5005396147, 5005431113, 5005566455, 5005621856, 5005629695, 5005752039, 5005762763, 5005856517, 5005206847, 5005210128, 5005675519, 5003956170, 5003969471, 5004087256, 5004263978, 5004263980, 5004285470, 5004318700, 5004372115, 5004421523, 5004869977, 5004893599, 5004943610, 5005258794, 5003416706, 5004511543, 5004813249, 5003913218, 5003978824, 5004044517, 5004200543, 5004203648, 5004307667, 5004308727, 5004346233, 5004381795, 5004384769, 5004413845, 5004458441, 5004475586, 5004475587, 5004515063, 5004593594, 5004597697, 5004663519, 5004792009, 5004953519, 5005218865, 5005267249, 5005283489, 5005315732, 5005343887, 5005398191, 5005448180, 5005452777, 5005457584, 5005515553, 5005539653, 5004955658, 5004956769, 5003252513, 5003903754, 5003903752, 5003903749, 5005275503, 5004350277], "supply_id": "WB-GI-235725654"}, {"account": "Старт", "order_ids": [5004116800, 5005574620, 5005619350, 5003887124, 5004105917, 5004698665, 5005664282, 5004763287, 5005636548, 5003696855, 5003133759, 5004575952, 5004852686, 5003940688, 5004269683], "supply_id": "WB-GI-235725645"}, {"account": "ВЕКТОР4216", "order_ids": [5004194054, 5004875600, 5004081756, 5004072106, 5004211197, 5004670921, 5004823956, 5005735928, 5005776357, 5003879856, 5004590323, 5003309218, 5005618450, 5005104579, 5005571946, 5004636049, 5004741717, 5003961812, 5003983590, 5004027552, 5004052603, 5004279613, 5004339850, 5004379218, 5004381899, 5004490082, 5004490080, 5004554169, 5004605855, 5004695609, 5004702675, 5004751774, 5004753905, 5004782140, 5004786390, 5004873223, 5004953287, 5004961582, 5004964527, 5004997238, 5005014794, 5005036223, 5005040294, 5005127111, 5005138202, 5005212301, 5005213402, 5005311894, 5005323301, 5005407157, 5005500745, 5005787097, 5005793882, 5004363105, 5004879093, 5004962426, 5005089283, 5003433621], "supply_id": "WB-GI-235725631"}, {"account": "ВЕКТОР7794", "order_ids": [5004363626, 5004282554, 5004549631, 5004818006, 5004922226, 5003867582, 5005044324, 5005066272, 5005516744, 5005137299, 5005175300, 5005297749, 5005352938, 5005355897, 5004284439, 5004008094, 5004008095, 5004008096, 5004141971, 5004798578, 5003891956, 5004564690, 5005732634, 5005848737], "supply_id": "WB-GI-235725647"}, {"account": "Даниелян", "order_ids": [5004515027, 5005021398, 5004036212, 5004074569, 5004162077, 5004162079, 5004275112, 5004349408, 5004356798, 5004371509, 5004371510, 5004386858, 5004406648, 5004561604, 5004737813, 5004760386, 5004855947, 5005028147, 5005171972, 5005220681, 5005318261, 5004260895, 5004260894, 5003428090, 5005757247, 5004389264, 5004559275, 5004821100, 5004869770, 5003890400, 5003932478, 5004020432, 5004162154, 5004255620, 5004577116, 5004679131, 5004679133, 5004690342, 5004823479, 5004881054, 5004924892, 5004955111, 5004970040, 5005019398, 5005032044, 5005037864, 5005291364, 5005474975, 5005503874, 5005779866, 5004384649, 5004388351, 5004491198, 5004806500, 5005267043, 5005662464, 5005353232], "supply_id": "WB-GI-235725653"}, {"account": "Старт2", "order_ids": [5005034059, 5004309989, 5004339795, 5004595792, 5005363966, 5003933672, 5003990981, 5004620830, 5004750740, 5004752248, 5005158186, 5005195654, 5005441350, 5005446212, 5005593013, 5004477996, 5004533908, 5004595358, 5004599370, 5004872220, 5005792712, 5004056289, 5005218136, 5005816405, 5003156431, 5004592944, 5004876311, 5005305016, 5005316610, 5005317262], "supply_id": "WB-GI-235725648"}, {"account": "СТАРТ9237", "order_ids": [5005693743, 5004238114, 5004655701, 5004655704, 5004927302, 5004042922, 5004479312, 5005034515, 5004445096, 5003720314, 5004281128, 5005026688, 5005806512, 5004702498, 5005126663, 5005239537, 5005241978, 5005242257, 5004259424, 5003256463, 5004269685, 5005724001, 5003520488, 5004482550], "supply_id": "WB-GI-235725640"}, {"account": "ВЕКТОР3522", "order_ids": [5005830418, 5004084384, 5003483342, 5004129554, 5004312362, 5005798733, 5005814985], "supply_id": "WB-GI-235725637"}, {"account": "СТАРТ7528", "order_ids": [5003244195, 5005583490, 5003654994, 5003654991, 5003654992, 5003654993, 5003654995, 5003672169, 5003912308, 5003969615, 5004139333, 5004590367, 5004805406, 5004805402, 5004805404, 5004805405, 5004805408, 5005021585, 5005041309, 5005146105, 5005146107, 5005146103, 5005146108, 5005146106, 5005263725, 5005323919, 5005323917, 5005516542, 5005539396, 5005539395, 5005782438, 5005638939], "supply_id": "WB-GI-235725634"}, {"account": "Оганесян", "order_ids": [5004062139, 5003918033, 5003931143, 5004147103, 5004182699, 5004497165, 5004524810, 5004648139, 5004715337, 5004983107, 5005661400, 5003945029, 5004120827, 5004193001, 5004339020, 5004397455, 5004483598, 5004668174, 5004739799, 5005045963, 5005084023, 5005084639, 5005119754, 5005214611, 5005594785, 5005629242, 5005645595, 5004012257, 5004086907, 5004668892, 5005529796, 5005627041, 5005671542, 5005687721, 5003857031, 5004309370, 5005806403, 5003180997, 5005272463, 5003883672, 5003883670, 5003883667, 5003883669, 5003883673, 5003883668, 5003883675, 5003883674, 5003883676, 5003883671, 5004254748, 5004254754, 5005709888], "supply_id": "WB-GI-235725643"}, {"account": "Вектор2", "order_ids": [5004379765, 5005449471, 5004045138, 5003880608, 5003907556, 5004129195, 5004182966, 5004276655, 5004285262, 5004430526, 5005440985, 5005440986, 5005477475, 5005506926, 5005698816, 5005790168, 5003985717, 5004395292, 5004930801, 5005220218, 5005494679, 5005577963, 5003855675, 5003865449, 5004897478, 5005072425, 5005680745, 5005009947, 5004271047, 5004942355, 5003169822, 5003995829, 5004239066, 5004518182, 5004881627, 5004996512, 5005261474, 5005306419, 5005538897, 5005555008, 5005843676, 5004185118], "supply_id": "WB-GI-235725652"}, {"account": "Вектор", "order_ids": [5003868369, 5004049506, 5004648564, 5005027341, 5005207852, 5005761822, 5005679347, 5004008402, 5004906406, 5005370297, 5003927115, 5003954346, 5004094994, 5004114034, 5004114035, 5004387262, 5004742710, 5004746679, 5004752165, 5005086654, 5005291912, 5005558306, 5004418818, 5004347918, 5004440885, 5004630990, 5005683700, 5003851773, 5003919954, 5003938898, 5003939982, 5003948334, 5003962545, 5003963400, 5003969856, 5004007871, 5004008598, 5004029248, 5004036006, 5004037507, 5004051743, 5004052702, 5004104287, 5004119542, 5004143797, 5004143796, 5004147862, 5004154069, 5004154274, 5004155759, 5004192834, 5004313455, 5004314758, 5004314756, 5004314757, 5004357606, 5004360245, 5004376109, 5004391599, 5004412584, 5004431772, 5004472948, 5004496730, 5004513772, 5004515917, 5004595116, 5004595785, 5004596118, 5004598289, 5004599790, 5004601770, 5004624861, 5004629362, 5004634072, 5004638518, 5004645274, 5004668115, 5004683067, 5004721061, 5004728017, 5004761824, 5004813965, 5004813963, 5004819686, 5004827861, 5004828411, 5004885244, 5004899644, 5004901855, 5004901856, 5004908141, 5004972611, 5004989073, 5004995512, 5004996135, 5005010252, 5005016984, 5005030789, 5005055429, 5005057051, 5005064477, 5005073999, 5005074358, 5005105498, 5005145942, 5005186795, 5005187188, 5005211572, 5005213147, 5005221338, 5005238711, 5005255943, 5005272799, 5005307465, 5005312290, 5005315736, 5005321577, 5005338344, 5005422841, 5005501753, 5005513923, 5005529152, 5005543558, 5005563669, 5005563810, 5005571907, 5005612851, 5005617926, 5005629871, 5005676251, 5005676250, 5005724805, 5005728981, 5005735947, 5005752560, 5005754302, 5005754301, 5005754300, 5005777045, 5005783216, 5005789215, 5005796000, 5005844455, 5003374985, 5003365040, 5005662520, 5004791600, 5003395246, 5004098101, 5004836424, 5004189945, 5004218181, 5004963469, 5004992645, 5003315599, 5005603876, 5004144020, 5004461632, 5003906722, 5003906723, 5003906720], "supply_id": "WB-GI-235725633"}, {"account": "ВЕКТОР5531", "order_ids": [5003908689, 5004433604, 5004478078, 5004478653, 5004507771, 5004579909, 5004753137, 5004753136, 5005152912, 5005346134, 5005349760, 5005606493, 5005817586, 5003924689, 5004051381, 5004079411, 5004569832, 5004803631, 5004904522, 5005178085, 5005183040, 5005576075, 5004914751, 5004189858, 5004356751, 5004459944, 5004478605, 5004564713, 5004775713, 5005256710, 5004014100, 5004640289, 5004068585, 5004450470, 5004452372, 5004902335, 5004929799, 5003958698, 5004153771, 5005051460, 5005140470, 5003953104, 5004611408, 5004856827, 5004856828, 5005015526, 5005269883, 5005388105, 5003376135], "supply_id": "WB-GI-235725636"}, {"account": "ВЕКТОР3745", "order_ids": [5005400299, 5004266683, 5004266685, 5004463371, 5004349434, 5005305915, 5004439011, 5004902952, 5004929785, 5003887516, 5004474103, 5005052103, 5005105524, 5005628879, 5005657643, 5005743569, 5004053423, 5004321487, 5005507364, 5003820373, 5003820374, 5003983899, 5003983900, 5003983902, 5004524617, 5004551769, 5004883777, 5005324591, 5004103408], "supply_id": "WB-GI-235725630"}, {"account": "Хачатрян", "order_ids": [5005749533, 5005381915, 5004733601, 5004647357, 5004917367, 5005006141, 5005035824, 5005485862, 5005781476, 5004076164, 5004450856, 5005410880, 5004078581, 5005341729, 5005361559, 5005361557, 5005361556, 5005361560, 5005361558, 5004603316, 5004603315, 5004777268, 5004844269, 5005236391, 5005689847, 5005692310], "supply_id": "WB-GI-235725638"}, {"account": "СТАРТ7409", "order_ids": [5003862173, 5003883281, 5004192919, 5004202105, 5004368020, 5005117573, 5005502955, 5005684909, 5005706232, 5005718995, 5004146691, 5004242941, 5004951566, 5004011784, 5004098382, 5004229327, 5004271333, 5005763761, 5003895960, 5004103684, 5004351385, 5004653095, 5004717440, 5004789069, 5004948917, 5005021665, 5005604814, 5005648142, 5005661728, 5005778757, 5003153719, 5003348326, 5004481399, 5004378025, 5004787551, 5005220838, 5005669993, 5003506479, 5003995103, 5005251488, 5005733430, 5005018595, 5004541890, 5004801627], "supply_id": "WB-GI-235725641"}, {"account": "ВЕКТОР8830", "order_ids": [5003911475, 5004892102, 5004419308, 5004711667, 5005447291, 5005851420, 5003816084, 5004097908, 5004132062, 5004133489, 5004176266, 5004207126, 5004233489, 5004354103, 5004440209, 5004551223, 5004560366, 5004560367, 5004567254, 5004567253, 5004611409, 5004932431, 5005210256, 5005336607, 5005381985, 5005392531, 5005437797, 5005437798, 5005629831], "supply_id": "WB-GI-235725644"}, {"account": "Тоноян", "order_ids": [5003995395, 5004063829, 5004168218, 5004306960, 5004441919, 5004486667, 5004519011, 5004703810, 5004982321, 5005023045, 5005059081, 5005458172, 5005777722, 5005831135, 5003914500, 5003927267, 5003954287, 5003987929, 5003991567, 5004003691, 5004017776, 5004036126, 5004067402, 5004067832, 5004107303, 5004207532, 5004275212, 5004288512, 5004291601, 5004294801, 5004299666, 5004309459, 5004343424, 5004461976, 5004464879, 5004466417, 5004470459, 5004569978, 5004598538, 5004664028, 5004719500, 5004843180, 5004896255, 5004896256, 5004946088, 5004974193, 5005001541, 5005177900, 5005183266, 5005235399, 5005309809, 5005345458, 5005381917, 5005416468, 5005456709, 5005490094, 5005553032, 5005600006, 5005600267, 5005634048, 5005644984, 5005696711, 5005700337, 5005737960, 5005776133, 5005848749, 5004528173, 5004569303, 5004582751, 5004909896, 5005118832, 5005541446, 5005642917, 5003996494, 5005581540, 5005710272, 5005755610, 5005790237, 5003895342, 5004715966, 5005729919, 5005191444, 5005825613, 5003897191, 5003913669, 5003913672, 5003953737, 5004011476, 5004107852, 5004123148, 5004238536, 5004380513, 5004405667, 5004615250, 5004640851, 5005060863, 5005108300, 5005194791, 5005204562, 5005211648, 5005256529, 5005322546, 5005375150, 5005420998, 5005460330, 5005519659, 5005625775, 5005737927, 5004245780, 5004437197, 5003800204, 5004839520, 5005382242, 5004906196, 5005274832, 5004326228, 5004665941, 5005072895, 5005583251, 5003499912], "supply_id": "WB-GI-235725642"}, {"account": "СТАРТ3925", "order_ids": [5004154307, 5004272692, 5004324454, 5005806561, 5003940602, 5004417526, 5004486922, 5004638016, 5004815044, 5005043981, 5005114980, 5005231070, 5005420863, 5005427830, 5005436331, 5005455046, 5005633216, 5005635593, 5005648782, 5005780094, 5004630754, 5005217229, 5004852703, 5005346608, 5004271245, 5004369050, 5004881348, 5005361928, 5003925430, 5003320457, 5004647360, 5003892530, 5005026687, 5005266722, 5005266723, 5005266726, 5005266724, 5005266728, 5005266725, 5005266727], "supply_id": "WB-GI-235725635"}, {"account": "СТАРТ8748", "order_ids": [5004134648, 5004490729, 5005274795, 5005399419, 5005464136, 5004085710, 5003999048, 5004289740, 5005456713, 5003892079, 5004025732, 5004241994, 5004293428, 5004541809, 5004560888, 5004585018, 5004604209, 5004649606, 5004887462, 5005036962, 5005192629, 5005251714, 5005440309, 5003539463, 5003774543, 5004807722, 5005451444, 5005577932, 5005642239, 5003853302, 5004392676, 5004393383, 5005282160, 5005282161, 5003206255, 5004052457], "supply_id": "WB-GI-235725646"}, {"account": "ВЕКТОР4135", "order_ids": [5004286982, 5004107636, 5004546361, 5004546362, 5004932507, 5005267095, 5005781323], "supply_id": "WB-GI-235725650"}, {"account": "СТАРТ5020", "order_ids": [5003800044, 5004030011, 5004041877, 5004399449, 5004517846, 5004517849], "supply_id": "WB-GI-235725632"}, {"account": "СТАРТ9900", "order_ids": [5004030012, 5004318092, 5004517847, 5004517848, 5004551767, 5004591105, 5004601414, 5005007784, 5005265848], "supply_id": "WB-GI-235725639"}, {"account": "ВЕКТОР8159", "order_ids": [5004551770], "supply_id": "WB-GI-235725649"}], "operation_id": "supply_1777960086_8ab4b6c5_fiz", "order_qr_map": {"5003133759": "50649447956", "5003153719": "50649466270", "5003156431": "50649468873", "5003169822": "51755387305", "5003180997": "50649491555", "5003206255": "50649515149", "5003244195": "50649550334", "5003252513": "50649557870", "5003256463": "50649561525", "5003309218": "51755447702", "5003315599": "51755453881", "5003320457": "53566454666", "5003348326": "53566482432", "5003365040": "53566499139", "5003374985": "53566509084", "5003376135": "53568671549", "5003395246": "53568690343", "5003416706": "53568710251", "5003428090": "53568721530", "5003433621": "53568727177", "5003460068": "53568753580", "5003483342": "53568776679", "5003499912": "53568793228", "5003506479": "53568799793", "5003520488": "53568813912", "5003537569": "53568830852", "5003539463": "53568832877", "5003654991": "53568948122", "5003654992": "53568948123", "5003654993": "53568948125", "5003654994": "53568948124", "5003654995": "53568948131", "5003672169": "53568965457", "5003696855": "53568989970", "5003720314": "53569013460", "5003774543": "53569067803", "5003800044": "53569093165", "5003800204": "53569093325", "5003816084": "53569109329", "5003820373": "53569113595", "5003820374": "53569113597", "5003851773": "53569144855", "5003853302": "53569146411", "5003855675": "53569148887", "5003857031": "53569150201", "5003857716": "53569150895", "5003862173": "53569155256", "5003865449": "53569158531", "5003867582": "53569160759", "5003868369": "53569161547", "5003869570": "53569162782", "5003879856": "53569172906", "5003880608": "53569173657", "5003883281": "53569176329", "5003883667": "53569178570", "5003883668": "53569178555", "5003883669": "53569178569", "5003883670": "53569178556", "5003883671": "53569178572", "5003883672": "53569178575", "5003883673": "53569178583", "5003883674": "53569178558", "5003883675": "53569178564", "5003883676": "53569178566", "5003887124": "53569180010", "5003887516": "53569180398", "5003890400": "53569185568", "5003891956": "53569184874", "5003892079": "53569184997", "5003892530": "53569185450", "5003892627": "53569185674", "5003895342": "53569188388", "5003895960": "53569189005", "5003897191": "53569190101", "5003903749": "53569196642", "5003903752": "53569196645", "5003903754": "53569196649", "5003906720": "53569199703", "5003906722": "53569199708", "5003906723": "53569199717", "5003907556": "53569200575", "5003908689": "53569201571", "5003911475": "53569204485", "5003912286": "53569205261", "5003912308": "53569205283", "5003913218": "53569206192", "5003913669": "53569206644", "5003913672": "53569206652", "5003914500": "53569207462", "5003915061": "53569208064", "5003918033": "53569211039", "5003919954": "53569212829", "5003921760": "53569214765", "5003924689": "53569217658", "5003925430": "53569218399", "5003927115": "53569219951", "5003927267": "53569220104", "5003931143": "53569224134", "5003932478": "53569225478", "5003933672": "53569226671", "5003938898": "53569231892", "5003939982": "53569232848", "5003940602": "53569233338", "5003940688": "53569233422", "5003945029": "53569237977", "5003948334": "53569241330", "5003953104": "53569245969", "5003953737": "53569246472", "5003954287": "53569247153", "5003954346": "53569247211", "5003956170": "53569249159", "5003958698": "53569251657", "5003961812": "53569254673", "5003962545": "53569255403", "5003963400": "53569256391", "5003969471": "53569262460", "5003969615": "53569262604", "5003969856": "53569262844", "5003978824": "53569271807", "5003983590": "53578438886", "5003983899": "53578441195", "5003983900": "53578441199", "5003983902": "53578441200", "5003985717": "53578443916", "5003987929": "53578450239", "5003990981": "53578457437", "5003991567": "53578460022", "5003995103": "53578467525", "5003995395": "53578467817", "5003995536": "53578467958", "5003995829": "53578469250", "5003996494": "53578469916", "5003999048": "53578478505", "5004003691": "53578490013", "5004007871": "53578500324", "5004008094": "53578500548", "5004008095": "53578500547", "5004008096": "53578500549", "5004008402": "53578500854", "5004008598": "53578503050", "5004009401": "53578503851", "5004011476": "53578508922", "5004011784": "53578510231", "5004012257": "53578510702", "5004014100": "53578515543", "5004017776": "53578525222", "5004020432": "53578530878", "5004025732": "53578546048", "5004025816": "53578546131", "5004027552": "53578549997", "5004029248": "53578553690", "5004030011": "53578555192", "5004030012": "53578555197", "5004036006": "53578570307", "5004036126": "53578570428", "5004036212": "53578570512", "5004037507": "53578573646", "5004041877": "53578585269", "5004042922": "53578588320", "5004044517": "53578591948", "5004045138": "53578593570", "5004049506": "53578604740", "5004051381": "53578609684", "5004051743": "53578611175", "5004052457": "53578611889", "5004052603": "53578614034", "5004052702": "53578614132", "5004053423": "53578614856", "5004056289": "53578621719", "5004062139": "53578636434", "5004063829": "53578641126", "5004067402": "53578649697", "5004067832": "53578651129", "5004068585": "53578651881", "5004072106": "53578662403", "5004074569": "53578667996", "5004076164": "53578672591", "5004078581": "53578677878", "5004079411": "53578679708", "5004081756": "53578687174", "5004084384": "53578692643", "5004085710": "53578694971", "5004086907": "53578699331", "5004087256": "53578699551", "5004089050": "53578705214", "5004090095": "53578707352", "5004094994": "53578719415", "5004097908": "53578727329", "5004098101": "53578727524", "5004098382": "53578727801", "5004103408": "53578740823", "5004103684": "53578742098", "5004104287": "53578749031", "5004105917": "53578746071", "5004107303": "53578749589", "5004107636": "53578749918", "5004107852": "53578751266", "5004111387": "53578759669", "5004114034": "53578766447", "5004114035": "53578766450", "5004116800": "53578774210", "5004118093": "53578777503", "5004119542": "53578779952", "5004120827": "53578782976", "5004123148": "53578789522", "5004129195": "53578813080", "5004129554": "53578805798", "5004129651": "53578805896", "5004132062": "53578820113", "5004133489": "53578815602", "5004134648": "53578818890", "5004138918": "53578830326", "5004139333": "53578830740", "5004141971": "53578838376", "5004143796": "53578843203", "5004143797": "53578843208", "5004143803": "53578843210", "5004144020": "53578843427", "5004146691": "53578848967", "5004147103": "53578850382", "5004147862": "53578853267", "5004153771": "53578867165", "5004154069": "53578867471", "5004154274": "53578867675", "5004154307": "53578867712", "5004155759": "53578872153", "5004160743": "53578884141", "5004162077": "53578887457", "5004162079": "53578887460", "5004162154": "53578887550", "5004168218": "53578901069", "5004176266": "53578924892", "5004182699": "53578940062", "5004182966": "53578940329", "5004185118": "53578944477", "5004189858": "53578956220", "5004189945": "53578956305", "5004192834": "53578960364", "5004192919": "53578960452", "5004193001": "53578960532", "5004194054": "53578961585", "5004200543": "53578968068", "5004202105": "53578969631", "5004203648": "53578971174", "5004207126": "53578974650", "5004207532": "53578975065", "5004211197": "53578978686", "5004213132": "53578980622", "5004218181": "53578985667", "5004225867": "53578993353", "5004229327": "53578996809", "5004233489": "53579001179", "5004238114": "53579001568", "5004238536": "53578904348", "5004239066": "53578904879", "5004241994": "53579007539", "5004242941": "53579008490", "5004245780": "53579011326", "5004254748": "53579020163", "5004254754": "53579020168", "5004255620": "53579021031", "5004259424": "53579024707", "5004260894": "53579026176", "5004260895": "53579026308", "5004263978": "53579029517", "5004263980": "53579029524", "5004266683": "53579032222", "5004266685": "53579032226", "5004269683": "53579035213", "5004269685": "53579035211", "5004271047": "53579036581", "5004271245": "53579036775", "5004271333": "53579036861", "5004272692": "53579038192", "5004275112": "53579040649", "5004275212": "53579040749", "5004276655": "53579044188", "5004279613": "53579045015", "5004281128": "53579046658", "5004282554": "53598016955", "5004284439": "53598018713", "5004285262": "53598019666", "5004285470": "53598019874", "5004286982": "53598021348", "5004288512": "53598022781", "5004289740": "53598024005", "5004291601": "53598025870", "5004293428": "53598027828", "5004294801": "53598029201", "5004299666": "53598034062", "5004304064": "53598038283", "5004306960": "53598041225", "5004307667": "53598041931", "5004308727": "53598043122", "5004309370": "53598043761", "5004309459": "53598043850", "5004309989": "53598044381", "5004312362": "53598046750", "5004313455": "53598050092", "5004314756": "53598048983", "5004314757": "53598048990", "5004314758": "53598048987", "5004318092": "53598052463", "5004318700": "53598053071", "5004321487": "53598055857", "5004324454": "53598061117", "5004326228": "53598060430", "5004339020": "53598072994", "5004339795": "53598073894", "5004339850": "53598073952", "5004343424": "53598077785", "5004346220": "53598080580", "5004346233": "53598080595", "5004347918": "53598082200", "5004349408": "53598083735", "5004349434": "53598083762", "5004350277": "53598084637", "5004351385": "53598085748", "5004354103": "53598088464", "5004356751": "53598091113", "5004356798": "53598091160", "5004357606": "53598091967", "5004360245": "53598094605", "5004363105": "53598097330", "5004363626": "53598097851", "5004368020": "53598102376", "5004369050": "53598103372", "5004371509": "53598105863", "5004371510": "53598105868", "5004372115": "53598106470", "5004374375": "53598108563", "5004376109": "53598110461", "5004378025": "53598112378", "5004379218": "53598113572", "5004379765": "53598114119", "5004380513": "53598114866", "5004381795": "53598116018", "5004381899": "53598116121", "5004384649": "53598119001", "5004384769": "53598119121", "5004386858": "53598121195", "5004387262": "53598121613", "5004388351": "53598122687", "5004389264": "53598123615", "5004391599": "53598125820", "5004392676": "53598127026", "5004393383": "53598127735", "5004395292": "53598129639", "5004397455": "53598131803", "5004399449": "53598133795", "5004405667": "53598140882", "5004406648": "53598141862", "5004412584": "53598147923", "5004413845": "53598149184", "5004417526": "53598152581", "5004418818": "53598153744", "5004419308": "53598154362", "5004421523": "53598156824", "5004430526": "53598165810", "5004431772": "53598167054", "5004433604": "53598168882", "5004435470": "53598170751", "5004437197": "53598172477", "5004439011": "53598174288", "5004440209": "53598175488", "5004440885": "53598176164", "5004441919": "53598177197", "5004445096": "53598180374", "5004447746": "53598182881", "5004450470": "53598185739", "5004450856": "53598186126", "5004452372": "53598187638", "5004458441": "53598193558", "5004459944": "53598195042", "5004461632": "53598196877", "5004461976": "53598197219", "5004463371": "53598198614", "5004464879": "53598200122", "5004466417": "53598201661", "5004470459": "53598205700", "5004472948": "53598208059", "5004474103": "53598209213", "5004475586": "53598210697", "5004475587": "53598210700", "5004477996": "53598213107", "5004478078": "53598213186", "5004478605": "53598213715", "5004478653": "53598213763", "5004479312": "53598214422", "5004481399": "53598216633", "5004482550": "53598217654", "5004483598": "53598218828", "5004486667": "53598221756", "5004486922": "53598222011", "5004488586": "53598223651", "5004490080": "53598225265", "5004490082": "53598225266", "5004490729": "53598225911", "5004491198": "53598226381", "5004496730": "53598231716", "5004497043": "53598232028", "5004497165": "53598232281", "5004498001": "53598233116", "5004507771": "53598242803", "5004511543": "53598246282", "5004513772": "53598248738", "5004515027": "53598249991", "5004515063": "53598252056", "5004515917": "53598250740", "5004516165": "53598250975", "5004517846": "53598252786", "5004517847": "53598252789", "5004517848": "53598252788", "5004517849": "53598252790", "5004518182": "53598252993", "5004519011": "53598253822", "5004519170": "53598253981", "5004523054": "53598257992", "5004524617": "53598259553", "5004524810": "53598259744", "5004528173": "53598263108", "5004531432": "53598266353", "5004533908": "53598268841", "5004534332": "53598269265", "5004539066": "53598273999", "5004541809": "53598276607", "5004541890": "53598276687", "5004546361": "53598281296", "5004546362": "53598281290", "5004547690": "53598282617", "5004549631": "53598284558", "5004551223": "53598286145", "5004551767": "53598286691", "5004551769": "53598286692", "5004551770": "53598286693", "5004554169": "53598289090", "5004556655": "53598291575", "5004559275": "53598294194", "5004560366": "53598295154", "5004560367": "53598295155", "5004560888": "53598295675", "5004561604": "53598296392", "5004564690": "53598299607", "5004564713": "53598299630", "5004565281": "53598300198", "5004567253": "53598302168", "5004567254": "53598302170", "5004569303": "53598304218", "5004569832": "53598304747", "5004569978": "53598304889", "5004575952": "53598310865", "5004577116": "53598312023", "5004579909": "53598314690", "5004582214": "53612624108", "5004582751": "53612624645", "5004585018": "53612626904", "5004590323": "53612632085", "5004590367": "53612632129", "5004591105": "53612632735", "5004592944": "53612634706", "5004593594": "53612635486", "5004595116": "53612637008", "5004595358": "53612637244", "5004595785": "53612637643", "5004595792": "53612637650", "5004596118": "53612637976", "5004597697": "53612639554", "5004598289": "53612640180", "5004598538": "53612640431", "5004599370": "53612641257", "5004599790": "53612641681", "5004601414": "53612643297", "5004601770": "53612643659", "5004603315": "53612645073", "5004603316": "53612645080", "5004604209": "53612645967", "5004605855": "53612647483", "5004607815": "53612649572", "5004611408": "53612653296", "5004611409": "53612653299", "5004613011": "53612654863", "5004615250": "53612657135", "5004620830": "53612662584", "5004621782": "53612663666", "5004624861": "53612666615", "5004629362": "53612671246", "5004630754": "53612672638", "5004630990": "53612672860", "5004634072": "53612678037", "5004636010": "53612677615", "5004636049": "53612677653", "5004638016": "53612679879", "5004638518": "53612680379", "5004640289": "53612682153", "5004640851": "53612682713", "5004645274": "53612687006", "5004647357": "53612689219", "5004647360": "53612689218", "5004648139": "53612689995", "5004648564": "53612690424", "5004649606": "53612691466", "5004653095": "53612694828", "5004655701": "53612697560", "5004655704": "53612697556", "5004663519": "53612705374", "5004664028": "53612705876", "5004665941": "53612707793", "5004668115": "53612709821", "5004668174": "53612709896", "5004668892": "53612710614", "5004670921": "53612712642", "5004674469": "53612716320", "5004679131": "53612720852", "5004679133": "53612720855", "5004683067": "53612724657", "5004690342": "53612732189", "5004695609": "53612737449", "5004698665": "53612740502", "5004702498": "53612744203", "5004702675": "53612744381", "5004703810": "53612745517", "5004708105": "53612749940", "5004708107": "53612752034", "5004708108": "53612749946", "5004708109": "53612752036", "5004708110": "53612752042", "5004708111": "53612752040", "5004708112": "53612752050", "5004708113": "53612752045", "5004708114": "53612752049", "5004708115": "53612752053", "5004711667": "53612753469", "5004715337": "53612757042", "5004715966": "53612757671", "5004717440": "53612759275", "5004719500": "53612761335", "5004721061": "53612762897", "5004725078": "53612766783", "5004728017": "53612769587", "5004733601": "53612775436", "5004737813": "53612779646", "5004739683": "53612781515", "5004739799": "53612781632", "5004741717": "53612786059", "5004742710": "53612784410", "5004746679": "53612788379", "5004750740": "53612792571", "5004751774": "53612793475", "5004752165": "53612793865", "5004752248": "53612793946", "5004753136": "53612794710", "5004753137": "53612794707", "5004753905": "53612795604", "5004760386": "53612802212", "5004761824": "53612803646", "5004763287": "53612805113", "5004775713": "53612817540", "5004777268": "53612819060", "5004782140": "53612823963", "5004784374": "53612826192", "5004785668": "53612827491", "5004786390": "53612828213", "5004787551": "53612829375", "5004789069": "53612830631", "5004791600": "53612833291", "5004792009": "53612833703", "5004797010": "53612838830", "5004798578": "53612840396", "5004801627": "53612843445", "5004803631": "53612845314", "5004805402": "53612847216", "5004805404": "53612847222", "5004805405": "53612847223", "5004805406": "53612847224", "5004805408": "53612847229", "5004806500": "53612848312", "5004807722": "53612849534", "5004813249": "53612854933", "5004813963": "53612855778", "5004813965": "53612855775", "5004815044": "53612856854", "5004818006": "53612862125", "5004819686": "53612861367", "5004821100": "53612862872", "5004823479": "53612865291", "5004823956": "53612865766", "5004827861": "53612869674", "5004828411": "53612870093", "5004836424": "53612878105", "5004838617": "53612883046", "5004839520": "53612881202", "5004843180": "53612884988", "5004844269": "53612886077", "5004852686": "53612894456", "5004852703": "53612894473", "5004854384": "53612896187", "5004855947": "53612897578", "5004856827": "53612898603", "5004856828": "53612898596", "5004869770": "53612911535", "5004869977": "53612911744", "5004872220": "53612914018", "5004873223": "53612915024", "5004874811": "53612916578", "5004875600": "53612917237", "5004876311": "53612917948", "5004879093": "53612920762", "5004881054": "53612922723", "5004881348": "53612923017", "5004881627": "53612923295", "5004883777": "53612925534", "5004885244": "53612927007", "5004887462": "53612929258", "5004892102": "53612933766", "5004893599": "53612935131", "5004896255": "53612937755", "5004896256": "53612937759", "5004897478": "53612939104", "5004899644": "53612941401", "5004901855": "53612943388", "5004901856": "53612943382", "5004902335": "53612943864", "5004902952": "53612944612", "5004904522": "53612946311", "5004906196": "53612947950", "5004906406": "53612948161", "5004908141": "53612949908", "5004909896": "53612951700", "5004914751": "53612956373", "5004917367": "53612959149", "5004922226": "53612964005", "5004924892": "53612966668", "5004927302": "53612969077", "5004929785": "53612971556", "5004929799": "53612971571", "5004930801": "53612972574", "5004932431": "53612973946", "5004932507": "53612974018", "5004942355": "53612984108", "5004943610": "53612985374", "5004946088": "53612990130", "5004948917": "53612990677", "5004951566": "53612993190", "5004953287": "53612995029", "5004953519": "53612995278", "5004955111": "53612996737", "5004955658": "53612997280", "5004956769": "53612998264", "5004961582": "53613003325", "5004962426": "53613004178", "5004963469": "53613005218", "5004964527": "53613006146", "5004970040": "53613011785", "5004972611": "53613014225", "5004974193": "53613015807", "5004982321": "53613023930", "5004983107": "53613024714", "5004989073": "53613033149", "5004992645": "53613034362", "5004995512": "53613037235", "5004996135": "53613037727", "5004996512": "53613038103", "5004997238": "53613038828", "5005001541": "53613043250", "5005004615": "53613046326", "5005006141": "53613047845", "5005007784": "53613049482", "5005009947": "53613051647", "5005010252": "53613051953", "5005014794": "53613056351", "5005015526": "53613057214", "5005016984": "53613058679", "5005018595": "53613060247", "5005019398": "53613061041", "5005021398": "53613062945", "5005021585": "53613063133", "5005021665": "53613063208", "5005023045": "53613064713", "5005026687": "53613068346", "5005026688": "53613068349", "5005027341": "53613069001", "5005028147": "53613069804", "5005030789": "53613072443", "5005032044": "53613073565", "5005033903": "53613075417", "5005034059": "53613075569", "5005034515": "53613076022", "5005035824": "53613077466", "5005036223": "53613077861", "5005036962": "53613078597", "5005037864": "53613079493", "5005040294": "53613081916", "5005041309": "53613082942", "5005043981": "53613085608", "5005044324": "53613085942", "5005045963": "53613087457", "5005051460": "53613092938", "5005052103": "53613093588", "5005055429": "53613097036", "5005057051": "53613098658", "5005059081": "53613100684", "5005060863": "53613102336", "5005064477": "53613106073", "5005065227": "53613106813", "5005066272": "53613107863", "5005072425": "53613114021", "5005072895": "53613114488", "5005073999": "53613115592", "5005074358": "53613115942", "5005084023": "53613125607", "5005084639": "53613126230", "5005086654": "53613127602", "5005089283": "53613133015", "5005092856": "53613136499", "5005104579": "53613148227", "5005105498": "53613149142", "5005105524": "53613149176", "5005108300": "53613151941", "5005114980": "53613158627", "5005117573": "53613161220", "5005118832": "53613162479", "5005119754": "53613163398", "5005126663": "53613170299", "5005127111": "53613170752", "5005137299": "53613180935", "5005138202": "53613181839", "5005140470": "53613184104", "5005145942": "53613189567", "5005146103": "53613189754", "5005146105": "53613189737", "5005146106": "53613189764", "5005146107": "53613189748", "5005146108": "53613189780", "5005152912": "53613196541", "5005158186": "53613201753", "5005171972": "53613213494", "5005175300": "53613216858", "5005177900": "53613219458", "5005178085": "53613219639", "5005178577": "53613220128", "5005183040": "53613224564", "5005183266": "53613224792", "5005186795": "53613228197", "5005187188": "53613228586", "5005191444": "53613232969", "5005192629": "53613234155", "5005194791": "53613236304", "5005195654": "53613237050", "5005204562": "53613246087", "5005206847": "53613248371", "5005206967": "53613248493", "5005207852": "53613249376", "5005210128": "53613251654", "5005210256": "53613251781", "5005211572": "53613253092", "5005211648": "53613253171", "5005212301": "53613253824", "5005213147": "53613254670", "5005213402": "53613254926", "5005214611": "53613256135", "5005217229": "53613258752", "5005218136": "53613259528", "5005218865": "53613260257", "5005220218": "53613261602", "5005220681": "53613262204", "5005220838": "53613262360", "5005221338": "53613262858", "5005226587": "53613268110", "5005231070": "53613272591", "5005235399": "53613276919", "5005236391": "53613277912", "5005236941": "53613278458", "5005238711": "53613280225", "5005239537": "53613280927", "5005241978": "53613283497", "5005242257": "53613283776", "5005245557": "53613286945", "5005245558": "53613286947", "5005245559": "53613286949", "5005247616": "53613289003", "5005251488": "53613293004", "5005251714": "53613293231", "5005255943": "53613297460", "5005256529": "53613298040", "5005256710": "53613298229", "5005258794": "53613300309", "5005261474": "53613302987", "5005263725": "53613305237", "5005265848": "53613307360", "5005266722": "53613308234", "5005266723": "53613308250", "5005266724": "53613308260", "5005266725": "53613308281", "5005266726": "53613308290", "5005266727": "53613308286", "5005266728": "53613308271", "5005267043": "53613308560", "5005267095": "53613308608", "5005267249": "53613308762", "5005269883": "53613311265", "5005272463": "53613313977", "5005272799": "53613314311", "5005274795": "53613316307", "5005274832": "53613316344", "5005275503": "53613317014", "5005282160": "53613323671", "5005282161": "53613323672", "5005283489": "53613324997", "5005291364": "53613332870", "5005291912": "53613333419", "5005297749": "53613339253", "5005305016": "53613346517", "5005305915": "53613347417", "5005306419": "53613347921", "5005307465": "53613348965", "5005308997": "53613350499", "5005309809": "53613351311", "5005311894": "53613353396", "5005312290": "53613353791", "5005315732": "53613357231", "5005315736": "53613357234", "5005316610": "53613358109", "5005317262": "53613358761", "5005318261": "53613359758", "5005321577": "53613363076", "5005322546": "53613364045", "5005323301": "53613364799", "5005323917": "53613368112", "5005323919": "53613368116", "5005324337": "53613368531", "5005324591": "53613368784", "5005334823": "53613378807", "5005336607": "53613380592", "5005338344": "53613382329", "5005339608": "53613383593", "5005341729": "53613385714", "5005343887": "53613387872", "5005345458": "53613389441", "5005346134": "53613390117", "5005346608": "53613390591", "5005349395": "53613393376", "5005349760": "53613393741", "5005352938": "53613396919", "5005353232": "53613397213", "5005355897": "53613399877", "5005361556": "53613405537", "5005361557": "53613405535", "5005361558": "53613405538", "5005361559": "53613405540", "5005361560": "53613405539", "5005361928": "53613405907", "5005363966": "53613407945", "5005368531": "53613412510", "5005370297": "53613414276", "5005375150": "53613419129", "5005381915": "53613425893", "5005381917": "53613425894", "5005381985": "53613425963", "5005382242": "53613426219", "5005388105": "53613432082", "5005392531": "53613436505", "5005396147": "53613440121", "5005398191": "53613442164", "5005399419": "53613443392", "5005400299": "53613444272", "5005407157": "53613451128", "5005410880": "53613454851", "5005416468": "53613460438", "5005420863": "53613464832", "5005420998": "53613464967", "5005422841": "53613466811", "5005427830": "53613471799", "5005431113": "53613475082", "5005436331": "53613480299", "5005437797": "53613481765", "5005437798": "53613481766", "5005440309": "53613484277", "5005440985": "53613484953", "5005440986": "53613484954", "5005441350": "53613485318", "5005446212": "53613490180", "5005447291": "53613491259", "5005448180": "53613492147", "5005449471": "53613493438", "5005449514": "53613493480", "5005451444": "53613495411", "5005452777": "53613496744", "5005455046": "53613499013", "5005456709": "53613500676", "5005456713": "53613500680", "5005457584": "53613501550", "5005458172": "53613502139", "5005460330": "53613504297", "5005464136": "53613508100", "5005474975": "53613518939", "5005477475": "53613521439", "5005480444": "53613524405", "5005485862": "53613529823", "5005490094": "53613534158", "5005494679": "53613538638", "5005500745": "53613544703", "5005501753": "53613545712", "5005502955": "53613546914", "5005503874": "53613547834", "5005506926": "53613550883", "5005507364": "53613551321", "5005513923": "53613557872", "5005515553": "53613559508", "5005516542": "53613560497", "5005516744": "53613560699", "5005519659": "53613563614", "5005521539": "53613565493", "5005526014": "53613569965", "5005529152": "53613573101", "5005529796": "53613573746", "5005538897": "53613582846", "5005539395": "53613583345", "5005539396": "53613583348", "5005539653": "53613583601", "5005541446": "53613585392", "5005543558": "53613587504", "5005553032": "53613596941", "5005555008": "53613598951", "5005558306": "53613602244", "5005559393": "53613603331", "5005563669": "53613607604", "5005563810": "53613607746", "5005566455": "53613610391", "5005571907": "53613615828", "5005571946": "53613615867", "5005574620": "53613618532", "5005576075": "53613619987", "5005577932": "53613621844", "5005577963": "53613621873", "5005581540": "53613625452", "5005583251": "53613627162", "5005583490": "53613627400", "5005593013": "53613636920", "5005594785": "53613638672", "5005600006": "53613643893", "5005600267": "53613644154", "5005603876": "53613647761", "5005604814": "53613648698", "5005606493": "53613650377", "5005612851": "53613656702", "5005617926": "53613661775", "5005618450": "53613662299", "5005619350": "53613663191", "5005621856": "53613665698", "5005625775": "53613669616", "5005627041": "53613670882", "5005627480": "53613671321", "5005628879": "53613672720", "5005629242": "53613673080", "5005629695": "53613673535", "5005629831": "53613673671", "5005629871": "53613673711", "5005633216": "53613677055", "5005634048": "53613677886", "5005635593": "53613679421", "5005636548": "53613680375", "5005638939": "53613682707", "5005642239": "53613686005", "5005642917": "53613686682", "5005643643": "53613687408", "5005644984": "53613688749", "5005645595": "53613689360", "5005648142": "53613691904", "5005648782": "53613692544", "5005652120": "53613695882", "5005657643": "53613701405", "5005660626": "53613704387", "5005661400": "53613705031", "5005661728": "53613705488", "5005662464": "53613706225", "5005662520": "53613706281", "5005664282": "53613708043", "5005669993": "53613713750", "5005671542": "53613715299", "5005675519": "53613719276", "5005676250": "53613720008", "5005676251": "53613720006", "5005679347": "53613723102", "5005680745": "53613724498", "5005683700": "53613727452", "5005684909": "53613728660", "5005687721": "53613731461", "5005689847": "53613733588", "5005692310": "53613736140", "5005693743": "53613737484", "5005696711": "53613740452", "5005698816": "53613742557", "5005700337": "53613744078", "5005706232": "53613749973", "5005709888": "53613753626", "5005710272": "53613753881", "5005718995": "53613762684", "5005721544": "53613765232", "5005724001": "53613767690", "5005724805": "53613768493", "5005728981": "53613772640", "5005729919": "53613773580", "5005732634": "53613776294", "5005733430": "53613776961", "5005735928": "53613779582", "5005735947": "53613779601", "5005737927": "53613781580", "5005737960": "53613781613", "5005743569": "53613787222", "5005749533": "53613793174", "5005752039": "53613795679", "5005752560": "53613796200", "5005754300": "53613797940", "5005754301": "53613797944", "5005754302": "53613797946", "5005755297": "53613798938", "5005755610": "53613799251", "5005757247": "53613800888", "5005761822": "53613805461", "5005762763": "53613806401", "5005763761": "53613807399", "5005776133": "53613819749", "5005776357": "53613819972", "5005777045": "53613820657", "5005777722": "53613821338", "5005778757": "53613822370", "5005779866": "53613823479", "5005780094": "53613823706", "5005781323": "53613824925", "5005781476": "53613825069", "5005782438": "53613826031", "5005783216": "53613826809", "5005787097": "53613830690", "5005789215": "53613832805", "5005790168": "53613833760", "5005790237": "53613833830", "5005792712": "53613836305", "5005793882": "53613837475", "5005796000": "53613839593", "5005796774": "53613840359", "5005798733": "53613842326", "5005806403": "53613849995", "5005806512": "53613850104", "5005806561": "53613850154", "5005814985": "53613858576", "5005816405": "53613859997", "5005817586": "53613861178", "5005825613": "53613869171", "5005830418": "53613873974", "5005831135": "53613874691", "5005843676": "53613887225", "5005844455": "53613888005", "5005848737": "53613892286", "5005848749": "53613892298", "5005851420": "53613894839", "5005852243": "53613895792", "5005856517": "53613900054"}, "order_wild_map": {"5003133759": "wild1752", "5003153719": "wild1592", "5003156431": "wild1704", "5003169822": "wild1675", "5003180997": "wild1630", "5003206255": "wild654", "5003244195": "wild1106", "5003252513": "wild300", "5003256463": "wild457", "5003309218": "wild1549", "5003315599": "wild252", "5003320457": "wild1639", "5003348326": "wild160", "5003365040": "wild1704", "5003374985": "wild1635", "5003376135": "wild359", "5003395246": "wild1971", "5003416706": "wild1594", "5003428090": "wild1591", "5003433621": "wild935", "5003460068": "wild1105", "5003483342": "wild1592", "5003499912": "wild351", "5003506479": "wild1607", "5003520488": "wild697", "5003537569": "wild1105", "5003539463": "wild1787", "5003654991": "wild1978", "5003654992": "wild1978", "5003654993": "wild1978", "5003654994": "wild1978", "5003654995": "wild1978", "5003672169": "wild1978", "5003696855": "wild1635", "5003720314": "wild1971", "5003774543": "wild1787", "5003800044": "wild1978", "5003800204": "wild1971", "5003816084": "wild1978", "5003820373": "wild1978", "5003820374": "wild1978", "5003851773": "wild161", "5003853302": "wild252", "5003855675": "wild1567", "5003857031": "wild1604", "5003857716": "wild150", "5003862173": "wild1398", "5003865449": "wild1567", "5003867582": "wild1398", "5003868369": "wild128", "5003869570": "wild1105", "5003879856": "wild151", "5003880608": "wild151", "5003883281": "wild1398", "5003883667": "wild312", "5003883668": "wild312", "5003883669": "wild312", "5003883670": "wild312", "5003883671": "wild312", "5003883672": "wild312", "5003883673": "wild312", "5003883674": "wild312", "5003883675": "wild312", "5003883676": "wild312", "5003887124": "wild1556", "5003887516": "wild161", "5003890400": "wild161", "5003891956": "wild457", "5003892079": "wild172", "5003892530": "wild1973", "5003892627": "wild1542", "5003895342": "wild1604", "5003895960": "wild1569", "5003897191": "wild1625", "5003903749": "wild312", "5003903752": "wild312", "5003903754": "wild312", "5003906720": "wild935", "5003906722": "wild935", "5003906723": "wild935", "5003907556": "wild151", "5003908689": "wild128", "5003911475": "wild1398", "5003912286": "wild1604", "5003912308": "wild1978", "5003913218": "wild161", "5003913669": "wild1625", "5003913672": "wild1625", "5003914500": "wild150", "5003915061": "wild1569", "5003918033": "wild1398", "5003919954": "wild161", "5003921760": "wild1542", "5003924689": "wild1398", "5003925430": "wild1607", "5003927115": "wild1567", "5003927267": "wild150", "5003931143": "wild1398", "5003932478": "wild161", "5003933672": "wild1556", "5003938898": "wild161", "5003939982": "wild161", "5003940602": "wild151", "5003940688": "wild457", "5003945029": "wild150", "5003948334": "wild161", "5003953104": "wild1978", "5003953737": "wild1625", "5003954287": "wild150", "5003954346": "wild1567", "5003956170": "wild1569", "5003958698": "wild1973", "5003961812": "wild161", "5003962545": "wild161", "5003963400": "wild161", "5003969471": "wild1569", "5003969615": "wild1978", "5003969856": "wild161", "5003978824": "wild161", "5003983590": "wild161", "5003983899": "wild1978", "5003983900": "wild1978", "5003983902": "wild1978", "5003985717": "wild1556", "5003987929": "wild150", "5003990981": "wild1556", "5003991567": "wild150", "5003995103": "wild1607", "5003995395": "wild1398", "5003995536": "wild151", "5003995829": "wild172", "5003996494": "wild1603", "5003999048": "wild1603", "5004003691": "wild150", "5004007871": "wild161", "5004008094": "wild1978", "5004008095": "wild1978", "5004008096": "wild1978", "5004008402": "wild150", "5004008598": "wild161", "5004009401": "wild1542", "5004011476": "wild1625", "5004011784": "wild1567", "5004012257": "wild1603", "5004014100": "wild1639", "5004017776": "wild150", "5004020432": "wild161", "5004025732": "wild172", "5004025816": "wild128", "5004027552": "wild161", "5004029248": "wild161", "5004030011": "wild1978", "5004030012": "wild1978", "5004036006": "wild161", "5004036126": "wild150", "5004036212": "wild128", "5004037507": "wild161", "5004041877": "wild1978", "5004042922": "wild1398", "5004044517": "wild161", "5004045138": "wild1398", "5004049506": "wild128", "5004051381": "wild1398", "5004051743": "wild161", "5004052457": "wild654", "5004052603": "wild161", "5004052702": "wild161", "5004053423": "wild172", "5004056289": "wild1607", "5004062139": "wild1106", "5004063829": "wild1398", "5004067402": "wild150", "5004067832": "wild150", "5004068585": "wild1971", "5004072106": "wild1398", "5004074569": "wild128", "5004076164": "wild1971", "5004078581": "wild1973", "5004079411": "wild1398", "5004081756": "wild128", "5004084384": "wild1542", "5004085710": "wild160", "5004086907": "wild1603", "5004087256": "wild1569", "5004089050": "wild1542", "5004090095": "wild1542", "5004094994": "wild1567", "5004097908": "wild1978", "5004098101": "wild1971", "5004098382": "wild1567", "5004103408": "wild697", "5004103684": "wild1569", "5004104287": "wild161", "5004105917": "wild1556", "5004107303": "wild150", "5004107636": "wild1978", "5004107852": "wild1625", "5004111387": "wild1542", "5004114034": "wild1567", "5004114035": "wild1567", "5004116800": "wild1105", "5004118093": "wild1542", "5004119542": "wild161", "5004120827": "wild151", "5004123148": "wild1625", "5004129195": "wild151", "5004129554": "wild161", "5004129651": "wild150", "5004132062": "wild1978", "5004133489": "wild1978", "5004134648": "wild1567", "5004138918": "wild1398", "5004139333": "wild1978", "5004141971": "wild1978", "5004143796": "wild161", "5004143797": "wild161", "5004143803": "wild1542", "5004144020": "wild654", "5004146691": "wild151", "5004147103": "wild1398", "5004147862": "wild161", "5004153771": "wild1973", "5004154069": "wild161", "5004154274": "wild161", "5004154307": "wild1398", "5004155759": "wild161", "5004160743": "wild1398", "5004162077": "wild128", "5004162079": "wild128", "5004162154": "wild161", "5004168218": "wild1398", "5004176266": "wild1978", "5004182699": "wild1398", "5004182966": "wild151", "5004185118": "wild312", "5004189858": "wild151", "5004189945": "wild1978", "5004192834": "wild161", "5004192919": "wild1398", "5004193001": "wild151", "5004194054": "wild1105", "5004200543": "wild161", "5004202105": "wild1398", "5004203648": "wild161", "5004207126": "wild1978", "5004207532": "wild150", "5004211197": "wild1398", "5004213132": "wild1569", "5004218181": "wild1978", "5004225867": "wild1105", "5004229327": "wild1567", "5004233489": "wild1978", "5004238114": "wild1106", "5004238536": "wild1625", "5004239066": "wild172", "5004241994": "wild172", "5004242941": "wild151", "5004245780": "wild1752", "5004254748": "wild312", "5004254754": "wild312", "5004255620": "wild161", "5004259424": "wild351", "5004260894": "wild1567", "5004260895": "wild1567", "5004263978": "wild1569", "5004263980": "wild1569", "5004266683": "wild1398", "5004266685": "wild1398", "5004269683": "wild457", "5004269685": "wild457", "5004271047": "wild1604", "5004271245": "wild1604", "5004271333": "wild1567", "5004272692": "wild1398", "5004275112": "wild128", "5004275212": "wild150", "5004276655": "wild151", "5004279613": "wild161", "5004281128": "wild1971", "5004282554": "wild128", "5004284439": "wild1639", "5004285262": "wild151", "5004285470": "wild1569", "5004286982": "wild1973", "5004288512": "wild150", "5004289740": "wild1604", "5004291601": "wild150", "5004293428": "wild172", "5004294801": "wild150", "5004299666": "wild150", "5004304064": "wild1542", "5004306960": "wild1398", "5004307667": "wild161", "5004308727": "wild161", "5004309370": "wild1604", "5004309459": "wild150", "5004309989": "wild1398", "5004312362": "wild161", "5004313455": "wild161", "5004314756": "wild161", "5004314757": "wild161", "5004314758": "wild161", "5004318092": "wild1978", "5004318700": "wild1569", "5004321487": "wild172", "5004324454": "wild1398", "5004326228": "wild312", "5004339020": "wild151", "5004339795": "wild1398", "5004339850": "wild161", "5004343424": "wild150", "5004346220": "wild1542", "5004346233": "wild161", "5004347918": "wild1603", "5004349408": "wild128", "5004349434": "wild151", "5004350277": "wild697", "5004351385": "wild1569", "5004354103": "wild1978", "5004356751": "wild1567", "5004356798": "wild128", "5004357606": "wild161", "5004360245": "wild161", "5004363105": "wild1973", "5004363626": "wild1105", "5004368020": "wild1398", "5004369050": "wild1604", "5004371509": "wild128", "5004371510": "wild128", "5004372115": "wild1569", "5004374375": "wild1542", "5004376109": "wild161", "5004378025": "wild1604", "5004379218": "wild161", "5004379765": "wild1106", "5004380513": "wild1625", "5004381795": "wild161", "5004381899": "wild161", "5004384649": "wild172", "5004384769": "wild161", "5004386858": "wild128", "5004387262": "wild1567", "5004388351": "wild172", "5004389264": "wild1604", "5004391599": "wild161", "5004392676": "wild312", "5004393383": "wild312", "5004395292": "wild1556", "5004397455": "wild151", "5004399449": "wild1978", "5004405667": "wild1625", "5004406648": "wild128", "5004412584": "wild161", "5004413845": "wild161", "5004417526": "wild151", "5004418818": "wild1594", "5004419308": "wild172", "5004421523": "wild1569", "5004430526": "wild151", "5004431772": "wild161", "5004433604": "wild128", "5004435470": "wild1398", "5004437197": "wild1752", "5004439011": "wild160", "5004440209": "wild1978", "5004440885": "wild1604", "5004441919": "wild1398", "5004445096": "wild1675", "5004447746": "wild1542", "5004450470": "wild1971", "5004450856": "wild1971", "5004452372": "wild1971", "5004458441": "wild161", "5004459944": "wild1567", "5004461632": "wild654", "5004461976": "wild150", "5004463371": "wild1398", "5004464879": "wild150", "5004466417": "wild150", "5004470459": "wild150", "5004472948": "wild161", "5004474103": "wild161", "5004475586": "wild161", "5004475587": "wild161", "5004477996": "wild1592", "5004478078": "wild128", "5004478605": "wild1567", "5004478653": "wild128", "5004479312": "wild1398", "5004481399": "wild1603", "5004482550": "wild697", "5004483598": "wild151", "5004486667": "wild1398", "5004486922": "wild151", "5004488586": "wild1542", "5004490080": "wild161", "5004490082": "wild161", "5004490729": "wild1567", "5004491198": "wild172", "5004496730": "wild161", "5004497043": "wild1592", "5004497165": "wild1398", "5004498001": "wild1973", "5004507771": "wild128", "5004511543": "wild1607", "5004513772": "wild161", "5004515027": "wild1105", "5004515063": "wild161", "5004515917": "wild161", "5004516165": "wild150", "5004517846": "wild1978", "5004517847": "wild1978", "5004517848": "wild1978", "5004517849": "wild1978", "5004518182": "wild172", "5004519011": "wild1398", "5004519170": "wild151", "5004523054": "wild1603", "5004524617": "wild1978", "5004524810": "wild1398", "5004528173": "wild1567", "5004531432": "wild1542", "5004533908": "wild1592", "5004534332": "wild1542", "5004539066": "wild1398", "5004541809": "wild172", "5004541890": "wild172", "5004546361": "wild1978", "5004546362": "wild1978", "5004547690": "wild1542", "5004549631": "wild128", "5004551223": "wild1978", "5004551767": "wild1978", "5004551769": "wild1978", "5004551770": "wild1978", "5004554169": "wild161", "5004556655": "wild1542", "5004559275": "wild1604", "5004560366": "wild1978", "5004560367": "wild1978", "5004560888": "wild172", "5004561604": "wild128", "5004564690": "wild457", "5004564713": "wild1567", "5004565281": "wild150", "5004567253": "wild1978", "5004567254": "wild1978", "5004569303": "wild1567", "5004569832": "wild1398", "5004569978": "wild150", "5004575952": "wild1787", "5004577116": "wild161", "5004579909": "wild128", "5004582214": "wild128", "5004582751": "wild1567", "5004585018": "wild172", "5004590323": "wild151", "5004590367": "wild1978", "5004591105": "wild1978", "5004592944": "wild172", "5004593594": "wild161", "5004595116": "wild161", "5004595358": "wild1592", "5004595785": "wild161", "5004595792": "wild1398", "5004596118": "wild161", "5004597697": "wild161", "5004598289": "wild161", "5004598538": "wild150", "5004599370": "wild1603", "5004599790": "wild161", "5004601414": "wild1978", "5004601770": "wild161", "5004603315": "wild1978", "5004603316": "wild1978", "5004604209": "wild172", "5004605855": "wild161", "5004607815": "wild1542", "5004611408": "wild1978", "5004611409": "wild1978", "5004613011": "wild1604", "5004615250": "wild1625", "5004620830": "wild1556", "5004621782": "wild1542", "5004624861": "wild161", "5004629362": "wild161", "5004630754": "wild1592", "5004630990": "wild1604", "5004634072": "wild161", "5004636010": "wild1398", "5004636049": "wild1604", "5004638016": "wild151", "5004638518": "wild161", "5004640289": "wild1639", "5004640851": "wild1625", "5004645274": "wild161", "5004647357": "wild172", "5004647360": "wild1639", "5004648139": "wild1398", "5004648564": "wild128", "5004649606": "wild172", "5004653095": "wild1569", "5004655701": "wild1106", "5004655704": "wild1106", "5004663519": "wild161", "5004664028": "wild150", "5004665941": "wild312", "5004668115": "wild161", "5004668174": "wild151", "5004668892": "wild1603", "5004670921": "wild1398", "5004674469": "wild128", "5004679131": "wild161", "5004679133": "wild161", "5004683067": "wild161", "5004690342": "wild161", "5004695609": "wild161", "5004698665": "wild1556", "5004702498": "wild1973", "5004702675": "wild161", "5004703810": "wild1398", "5004708105": "wild1542", "5004708107": "wild1542", "5004708108": "wild1542", "5004708109": "wild1542", "5004708110": "wild1542", "5004708111": "wild1542", "5004708112": "wild1542", "5004708113": "wild1542", "5004708114": "wild1542", "5004708115": "wild1542", "5004711667": "wild172", "5004715337": "wild1398", "5004715966": "wild1604", "5004717440": "wild1569", "5004719500": "wild150", "5004721061": "wild161", "5004725078": "wild1105", "5004728017": "wild161", "5004733601": "wild1604", "5004737813": "wild128", "5004739683": "wild1603", "5004739799": "wild151", "5004741717": "wild1604", "5004742710": "wild1567", "5004746679": "wild1567", "5004750740": "wild1556", "5004751774": "wild161", "5004752165": "wild1567", "5004752248": "wild1556", "5004753136": "wild128", "5004753137": "wild128", "5004753905": "wild161", "5004760386": "wild128", "5004761824": "wild161", "5004763287": "wild1592", "5004775713": "wild1567", "5004777268": "wild1978", "5004782140": "wild161", "5004784374": "wild1542", "5004785668": "wild1542", "5004786390": "wild161", "5004787551": "wild1604", "5004789069": "wild1569", "5004791600": "wild1787", "5004792009": "wild161", "5004797010": "wild1542", "5004798578": "wild1978", "5004801627": "wild1752", "5004803631": "wild1398", "5004805402": "wild1978", "5004805404": "wild1978", "5004805405": "wild1978", "5004805406": "wild1978", "5004805408": "wild1978", "5004806500": "wild172", "5004807722": "wild1787", "5004813249": "wild1607", "5004813963": "wild161", "5004813965": "wild161", "5004815044": "wild151", "5004818006": "wild128", "5004819686": "wild161", "5004821100": "wild1604", "5004823479": "wild161", "5004823956": "wild1398", "5004827861": "wild161", "5004828411": "wild161", "5004836424": "wild1973", "5004838617": "wild150", "5004839520": "wild1973", "5004843180": "wild150", "5004844269": "wild1978", "5004852686": "wild1787", "5004852703": "wild1603", "5004854384": "wild128", "5004855947": "wild128", "5004856827": "wild1978", "5004856828": "wild1978", "5004869770": "wild1604", "5004869977": "wild1569", "5004872220": "wild1603", "5004873223": "wild161", "5004874811": "wild1542", "5004875600": "wild1105", "5004876311": "wild252", "5004879093": "wild1978", "5004881054": "wild161", "5004881348": "wild1604", "5004881627": "wild172", "5004883777": "wild1978", "5004885244": "wild161", "5004887462": "wild172", "5004892102": "wild1398", "5004893599": "wild1569", "5004896255": "wild150", "5004896256": "wild150", "5004897478": "wild1567", "5004899644": "wild161", "5004901855": "wild161", "5004901856": "wild161", "5004902335": "wild1971", "5004902952": "wild160", "5004904522": "wild1398", "5004906196": "wild1978", "5004906406": "wild151", "5004908141": "wild161", "5004909896": "wild1567", "5004914751": "wild150", "5004917367": "wild172", "5004922226": "wild128", "5004924892": "wild161", "5004927302": "wild1106", "5004929785": "wild160", "5004929799": "wild1971", "5004930801": "wild1556", "5004932431": "wild1978", "5004932507": "wild1978", "5004942355": "wild1604", "5004943610": "wild1569", "5004946088": "wild150", "5004948917": "wild1569", "5004951566": "wild151", "5004953287": "wild161", "5004953519": "wild161", "5004955111": "wild161", "5004955658": "wild1978", "5004956769": "wild1978", "5004961582": "wild161", "5004962426": "wild1978", "5004963469": "wild1978", "5004964527": "wild161", "5004970040": "wild161", "5004972611": "wild161", "5004974193": "wild150", "5004982321": "wild1398", "5004983107": "wild1398", "5004989073": "wild161", "5004992645": "wild1978", "5004995512": "wild161", "5004996135": "wild161", "5004996512": "wild172", "5004997238": "wild161", "5005001541": "wild150", "5005004615": "wild128", "5005006141": "wild172", "5005007784": "wild1978", "5005009947": "wild1603", "5005010252": "wild161", "5005014794": "wild161", "5005015526": "wild1978", "5005016984": "wild161", "5005018595": "wild161", "5005019398": "wild161", "5005021398": "wild1105", "5005021585": "wild1978", "5005021665": "wild1569", "5005023045": "wild1398", "5005026687": "wild1973", "5005026688": "wild1971", "5005027341": "wild128", "5005028147": "wild128", "5005030789": "wild161", "5005032044": "wild161", "5005033903": "wild151", "5005034059": "wild1105", "5005034515": "wild151", "5005035824": "wild172", "5005036223": "wild161", "5005036962": "wild172", "5005037864": "wild161", "5005040294": "wild161", "5005041309": "wild1978", "5005043981": "wild151", "5005044324": "wild1556", "5005045963": "wild151", "5005051460": "wild1973", "5005052103": "wild161", "5005055429": "wild161", "5005057051": "wild161", "5005059081": "wild1398", "5005060863": "wild1625", "5005064477": "wild161", "5005065227": "wild1542", "5005066272": "wild1603", "5005072425": "wild1567", "5005072895": "wild312", "5005073999": "wild161", "5005074358": "wild161", "5005084023": "wild151", "5005084639": "wild151", "5005086654": "wild1567", "5005089283": "wild1978", "5005092856": "wild128", "5005104579": "wild1569", "5005105498": "wild161", "5005105524": "wild161", "5005108300": "wild1625", "5005114980": "wild151", "5005117573": "wild1398", "5005118832": "wild1567", "5005119754": "wild151", "5005126663": "wild1978", "5005127111": "wild161", "5005137299": "wild1604", "5005138202": "wild161", "5005140470": "wild1973", "5005145942": "wild161", "5005146103": "wild1978", "5005146105": "wild1978", "5005146106": "wild1978", "5005146107": "wild1978", "5005146108": "wild1978", "5005152912": "wild128", "5005158186": "wild1556", "5005171972": "wild128", "5005175300": "wild161", "5005177900": "wild150", "5005178085": "wild1398", "5005178577": "wild1542", "5005183040": "wild1398", "5005183266": "wild150", "5005186795": "wild161", "5005187188": "wild161", "5005191444": "wild161", "5005192629": "wild172", "5005194791": "wild1625", "5005195654": "wild1556", "5005204562": "wild1625", "5005206847": "wild1556", "5005206967": "wild1542", "5005207852": "wild128", "5005210128": "wild1556", "5005210256": "wild1978", "5005211572": "wild161", "5005211648": "wild1625", "5005212301": "wild161", "5005213147": "wild161", "5005213402": "wild161", "5005214611": "wild151", "5005217229": "wild1592", "5005218136": "wild1607", "5005218865": "wild161", "5005220218": "wild1556", "5005220681": "wild128", "5005220838": "wild1604", "5005221338": "wild161", "5005226587": "wild128", "5005231070": "wild151", "5005235399": "wild150", "5005236391": "wild1978", "5005236941": "wild1542", "5005238711": "wild161", "5005239537": "wild1978", "5005241978": "wild1978", "5005242257": "wild1978", "5005245557": "wild128", "5005245558": "wild128", "5005245559": "wild128", "5005247616": "wild1542", "5005251488": "wild1607", "5005251714": "wild172", "5005255943": "wild161", "5005256529": "wild1625", "5005256710": "wild1567", "5005258794": "wild1569", "5005261474": "wild172", "5005263725": "wild1978", "5005265848": "wild1978", "5005266722": "wild1978", "5005266723": "wild1978", "5005266724": "wild1978", "5005266725": "wild1978", "5005266726": "wild1978", "5005266727": "wild1978", "5005266728": "wild1978", "5005267043": "wild172", "5005267095": "wild1978", "5005267249": "wild161", "5005269883": "wild1978", "5005272463": "wild1978", "5005272799": "wild161", "5005274795": "wild1567", "5005274832": "wild1978", "5005275503": "wild312", "5005282160": "wild312", "5005282161": "wild312", "5005283489": "wild161", "5005291364": "wild161", "5005291912": "wild1567", "5005297749": "wild161", "5005305016": "wild252", "5005305915": "wild151", "5005306419": "wild172", "5005307465": "wild161", "5005308997": "wild151", "5005309809": "wild150", "5005311894": "wild161", "5005312290": "wild161", "5005315732": "wild161", "5005315736": "wild161", "5005316610": "wild312", "5005317262": "wild312", "5005318261": "wild128", "5005321577": "wild161", "5005322546": "wild1625", "5005323301": "wild161", "5005323917": "wild1978", "5005323919": "wild1978", "5005324337": "wild128", "5005324591": "wild1978", "5005334823": "wild1542", "5005336607": "wild1978", "5005338344": "wild161", "5005339608": "wild1604", "5005341729": "wild1973", "5005343887": "wild161", "5005345458": "wild150", "5005346134": "wild128", "5005346608": "wild1603", "5005349395": "wild151", "5005349760": "wild128", "5005352938": "wild161", "5005353232": "wild1978", "5005355897": "wild161", "5005361556": "wild1973", "5005361557": "wild1973", "5005361558": "wild1973", "5005361559": "wild1973", "5005361560": "wild1973", "5005361928": "wild1604", "5005363966": "wild151", "5005368531": "wild1603", "5005370297": "wild1556", "5005375150": "wild1625", "5005381915": "wild150", "5005381917": "wild150", "5005381985": "wild1978", "5005382242": "wild1973", "5005388105": "wild1978", "5005392531": "wild1978", "5005396147": "wild1542", "5005398191": "wild161", "5005399419": "wild1567", "5005400299": "wild128", "5005407157": "wild161", "5005410880": "wild1971", "5005416468": "wild150", "5005420863": "wild151", "5005420998": "wild1625", "5005422841": "wild161", "5005427830": "wild151", "5005431113": "wild1542", "5005436331": "wild151", "5005437797": "wild1978", "5005437798": "wild1978", "5005440309": "wild1752", "5005440985": "wild151", "5005440986": "wild151", "5005441350": "wild1556", "5005446212": "wild1556", "5005447291": "wild172", "5005448180": "wild161", "5005449471": "wild1106", "5005449514": "wild1973", "5005451444": "wild1787", "5005452777": "wild161", "5005455046": "wild151", "5005456709": "wild150", "5005456713": "wild161", "5005457584": "wild161", "5005458172": "wild1398", "5005460330": "wild1625", "5005464136": "wild1567", "5005474975": "wild161", "5005477475": "wild151", "5005480444": "wild128", "5005485862": "wild172", "5005490094": "wild150", "5005494679": "wild1556", "5005500745": "wild161", "5005501753": "wild161", "5005502955": "wild1398", "5005503874": "wild161", "5005506926": "wild151", "5005507364": "wild172", "5005513923": "wild161", "5005515553": "wild161", "5005516542": "wild1978", "5005516744": "wild1603", "5005519659": "wild1625", "5005521539": "wild1973", "5005526014": "wild150", "5005529152": "wild161", "5005529796": "wild1603", "5005538897": "wild172", "5005539395": "wild1978", "5005539396": "wild1978", "5005539653": "wild161", "5005541446": "wild1567", "5005543558": "wild161", "5005553032": "wild150", "5005555008": "wild172", "5005558306": "wild1567", "5005559393": "wild151", "5005563669": "wild161", "5005563810": "wild161", "5005566455": "wild1542", "5005571907": "wild161", "5005571946": "wild160", "5005574620": "wild1398", "5005576075": "wild1398", "5005577932": "wild1787", "5005577963": "wild1556", "5005581540": "wild1603", "5005583251": "wild312", "5005583490": "wild1973", "5005593013": "wild1556", "5005594785": "wild151", "5005600006": "wild150", "5005600267": "wild150", "5005603876": "wild312", "5005604814": "wild1569", "5005606493": "wild128", "5005612851": "wild161", "5005617926": "wild161", "5005618450": "wild1556", "5005619350": "wild1398", "5005621856": "wild1542", "5005625775": "wild1625", "5005627041": "wild1603", "5005627480": "wild151", "5005628879": "wild161", "5005629242": "wild151", "5005629695": "wild1542", "5005629831": "wild1978", "5005629871": "wild161", "5005633216": "wild151", "5005634048": "wild150", "5005635593": "wild151", "5005636548": "wild161", "5005638939": "wild654", "5005642239": "wild1787", "5005642917": "wild1567", "5005643643": "wild151", "5005644984": "wild150", "5005645595": "wild151", "5005648142": "wild1569", "5005648782": "wild151", "5005652120": "wild151", "5005657643": "wild161", "5005660626": "wild151", "5005661400": "wild1398", "5005661728": "wild1569", "5005662464": "wild1971", "5005662520": "wild172", "5005664282": "wild1556", "5005669993": "wild1604", "5005671542": "wild1603", "5005675519": "wild1556", "5005676250": "wild161", "5005676251": "wild161", "5005679347": "wild1398", "5005680745": "wild1567", "5005683700": "wild1604", "5005684909": "wild1398", "5005687721": "wild1603", "5005689847": "wild1978", "5005692310": "wild1978", "5005693743": "wild1105", "5005696711": "wild150", "5005698816": "wild151", "5005700337": "wild150", "5005706232": "wild1398", "5005709888": "wild312", "5005710272": "wild1603", "5005718995": "wild1398", "5005721544": "wild1398", "5005724001": "wild457", "5005724805": "wild161", "5005728981": "wild161", "5005729919": "wild1604", "5005732634": "wild457", "5005733430": "wild1607", "5005735928": "wild1398", "5005735947": "wild161", "5005737927": "wild1625", "5005737960": "wild150", "5005743569": "wild161", "5005749533": "wild128", "5005752039": "wild1542", "5005752560": "wild161", "5005754300": "wild161", "5005754301": "wild161", "5005754302": "wild161", "5005755297": "wild1398", "5005755610": "wild1603", "5005757247": "wild1591", "5005761822": "wild128", "5005762763": "wild1542", "5005763761": "wild1567", "5005776133": "wild150", "5005776357": "wild1398", "5005777045": "wild161", "5005777722": "wild1398", "5005778757": "wild1569", "5005779866": "wild161", "5005780094": "wild151", "5005781323": "wild1978", "5005781476": "wild172", "5005782438": "wild1978", "5005783216": "wild161", "5005787097": "wild161", "5005789215": "wild161", "5005790168": "wild151", "5005790237": "wild1603", "5005792712": "wild1603", "5005793882": "wild161", "5005796000": "wild161", "5005796774": "wild1398", "5005798733": "wild161", "5005806403": "wild1604", "5005806512": "wild1971", "5005806561": "wild1398", "5005814985": "wild161", "5005816405": "wild161", "5005817586": "wild128", "5005825613": "wild161", "5005830418": "wild1105", "5005831135": "wild1398", "5005843676": "wild172", "5005844455": "wild161", "5005848737": "wild457", "5005848749": "wild150", "5005851420": "wild1973", "5005852243": "wild128", "5005856517": "wild1542"}}

In [5]:
client = SimaLandClient()
items = client.get_sim_land_items(82958)

if items:
    print(f"Первый товар: {items[0]['name']} - {items[0]['price']} руб.")

🚀 Начинаю выгрузку товаров для категории 82958...
✅ Загружено товаров: 136 (последний ID: 8215184)
Выгрузка завершена. Итого: 136 шт.
Первый товар: Защита роликовая ONLYTOP, р. S, цвет розовый - 249 руб.


In [9]:
df = pd.DataFrame(items)
df_filter = df[df['name'].str.contains("самокат", case=False, na=False)]
df_filter.head()

,sid,name,balance,price,price_max,boxtype_id,box_depth,box_width,box_height,width,height,per_package,photo_url
16,1738611,"Самокат городской GRAFFITI Technology 200, складной, колёса PU 200 мм, ABEC 7, амортизатор задний, чёрный",0,5999.0,5999,576,85.0,15.0,32.0,13.0,100,1,https://goods-photos.static1-sima-land.com/items/2058573/0/700.jpg?v=1740736479
28,5358842,"Самокат 2 в 1 GRAFFITI Kids, с корзинкой, колёса PU 120/75 мм, цвет серый",0,1699.0,1999,576,56.0,15.0,27.0,0.0,0,1,https://goods-photos.static1-sima-land.com/items/5744751/0/700.jpg?v=1758777032
29,5358843,"Самокат 2 в 1 с корзинкой GRAFFITI Kids, колёса PU 120/75 мм, розовый",0,2250.0,2250,576,56.0,15.0,27.0,0.0,0,1,https://goods-photos.static1-sima-land.com/items/5744752/0/700.jpg?v=1740981727
41,6913028,"Лыжи для самокатов-снегокатов GRAFFITI, пара, цвет розовый",12,449.0,449,574,38.0,9.0,18.0,9.0,18,1,https://goods-photos.static1-sima-land.com/items/6354827/0/700.jpg?v=1756210068
43,7339246,"Самокат GRAFFITI, складной, колёса PU 145 мм, ABEC 7, амортизатор передний, алюминиевый",11,3999.0,3999,576,68.0,11.0,26.0,0.0,0,1,https://goods-photos.static1-sima-land.com/items/6638137/0/700.jpg?v=1740982523


In [29]:
sima = SimaLandClient()
token = sima.token

category_id = 57332

headers = {
    "x-api-key": token
}
url = "https://www.sima-land.ru/api/v3/item/"
params_item = {
    "category_id": category_id,
    "has_balance": 1,
    "is_disabled": 0,
    "is_deleted": 0,
    "has_photo": 1,
}


url_category = "https://www.sima-land.ru/api/v3/category/"
name = "самокаты"
params_cat = {
    "id": category_id,
    "expand-root": {
        "level": 1
    }
}
url_attr = "https://www.sima-land.ru/api/v3/attr/"

In [23]:
path = r"sima_categories.json"
path_to_save = 'sima_goods.xlsx'
df_to_excel = pd.read_json(path)
df_to_excel = df_to_excel.to_excel(path_to_save)
df_to_excel.head()

AttributeError: 'NoneType' object has no attribute 'head'

In [15]:

headers = {"x-api-key": token}

# 1. Ищем все подкатегории внутри "Спорт и туризм" (level=2)
# Или ищем категорию, где в названии есть "самокаты"
url_category = "https://www.sima-land.ru/api/v3/category/"
params_cat = {
    "level": 5,          # Ищем на уровень ниже
    "path": "16",        # Указываем, что родитель - Спорт (ID 16)
    "is_active": 1
}

res = requests.get(url_category, headers=headers, params=params_cat)
categories = res.json().get('items', [])

In [16]:
for cat in categories:
    if "самокат" in cat['name'].lower():
        print(f"Найдена категория: {cat['name']} (ID: {cat['id']})")
    # else:
    #     print(f"Категория не подходит: {cat['name']} (ID: {cat['id']})")

Найдена категория: Самокаты и аксессуары (ID: 2208)


In [58]:
names = []
for item in data['items']:
    names.append((item['name'], item['id']))
print(names)

[('Аксессуары для волос', 1), ('Бижутерия', 2), ('Бизнес-сувениры', 3), ('Брелоки и подвески', 4), ('Талисманы и фэншуй', 5), ('Интерьерные сувениры', 7), ('Канцтовары', 8), ('Магниты и значки', 10), ('Подарочные наборы', 11), ('Посуда', 12), ('Приколы', 14), ('Спорт и туризм', 16), ('Сувениры с символикой стран и городов', 18), ('Оформление и организация праздника', 19), ('Подарочная упаковка', 20), ('Фоторамки и фотоальбомы', 21), ('Хозтовары', 22), ('Ободки', 26), ('Невидимки, зажимы и шпильки', 33), ('Резинки', 35), ('Крабы, бананы, заколки', 43), ('Наборы аксессуаров для волос', 50), ('Наклейки', 77), ('На телефон', 78), ('Термопосуда', 80), ('Туризм', 81), ('Термопосуда и термосумки', 82), ('Профессиональные праздники', 84), ('Военный', 85), ('Платки', 87), ('Ручки', 91), ('Ручки-приколы', 92), ('Стержни', 94), ('Карандаши', 96), ('Маркеры', 97), ('Визитницы', 98), ('Зеркала', 99), ('Наборы и подставки', 119), ('Блокноты и записные книжки', 121), ('Ежедневники', 124), ('Папки, си

In [ ]:
'wild1977',
'wild1972',
'wild1973',
'wild1969',
'wild1980',
'wild2083',
'wild1976',
'wild1983',
'wild1974',
'wild1982',
'wild1970',
'wild1978',
'wild1975',
'wild1979',
'wild2082',
'wild2132'